# MeChess 13/13: Elo of the Stockfish ladder and of the human-like bot (CPU notebook)

An adapter around `chessme humanlike-arena` (track S, S1 of the research plan). Stockfish at several **node limits** (the ladder) and the **human-like bot** (a Maia-3 model's 16 most probable moves, each judged by Stockfish, the move drawn from prior x exp(-beta x loss)) play each other from the opening moves of real games (both colours, same openings for every pair), without clocks, adjudicated by a fixed referee engine. The result is an Elo for every player relative to the weakest ladder player, with intervals that resample openings. It turns the "expected loss" axis of `humanlike-frontier` into Elo on the ladder's scale (absolute Lichess ratings need games against humans).

**Before you run:** add the output of notebook 10 (`games_human.jsonl`, for the openings) as an input; the Maia-3 weights and Stockfish are downloaded by this notebook (external downloads: AGPL-3.0 and GPL-3.0, not redistributed); *Accelerator -> None*, *Internet -> On*; set `NODES`, `BETAS`, `STARTS`; **Save Version -> Save & Run All (Commit)**. Bot games are the cost (about 17 engine searches per move). The run is **resumable**: finished games are appended to `games_arena.jsonl` and skipped when the same output is run again.

In [ ]:
import glob, os, shutil, subprocess, sys

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here (not needed when the repository is attached as a dataset)
WORKERS = 4
MODEL = "23m"                                           # which Maia-3 size the bot is built on: 5m, 23m or 79m
NODES = "1,10,30,100,300,1000,3000,10000"               # the Stockfish ladder: node limits (the weakest is the anchor, Elo 0)
BETAS = "0,3,10,30"                                     # the bot: weights of the engine's loss (0 = the human prior; larger = stronger)
BOT_NODES = 5000                                        # the bot's judge: nodes per candidate move
RATING = 2000                                           # the rating the bot tells the Maia model it plays at
STARTS = 24                                             # openings (each played with both colours against every opponent)
MAIA3_COMMIT = ""                                       # empty: the current default branch; a commit hash pins the baseline exactly
OUT = "/kaggle/working/humanlike" if os.path.exists("/kaggle") else "humanlike_local"
STOCKFISH_URL = "https://github.com/official-stockfish/Stockfish/releases/download/sf_18/stockfish-ubuntu-x86-64-avx2.tar"
INPUT_DIR = ""                                          # the output folder of notebook 10 (games_human.jsonl); empty: search /kaggle/input

def sh(*args):
    """Run a command and stream its output; stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):
    attached = sorted(glob.glob("/kaggle/input/**/chessme/__init__.py", recursive=True), key=lambda f: "/datasets/" not in f)   # prefer the attached repository dataset
    if attached:
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), "MeChess")
    else:                                                           # Kaggle: fetch the repository (Internet must be on)
        if "<you>" in REPO_URL:
            raise SystemExit("Attach the repository as a dataset, or set REPO_URL.")
        sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "chess==1.11.2", "numpy", "pyyaml", "requests", "zstandard")
CLI = [sys.executable, "-m", "chessme"]              # every step below is one `chessme` command: the same ones you run on a laptop

In [ ]:
root = INPUT_DIR or "/kaggle/input"
HUMAN = next(iter(sorted(glob.glob(root + "/**/games_human.jsonl", recursive=True))), None)
print(HUMAN)
if not HUMAN:
    raise SystemExit("Add the output of notebook 10 (games_human.jsonl) as an input.")
os.makedirs(OUT, exist_ok=True)

## Step 1. Stockfish and Maia-3 (external, not part of this repository)

In [ ]:
os.makedirs("/tmp/sf", exist_ok=True)
sh("curl", "-fsSL", "-o", "/tmp/sf/sf.tar", STOCKFISH_URL)
sh("tar", "-xf", "/tmp/sf/sf.tar", "-C", "/tmp/sf")
ENGINE = next(f for f in glob.glob("/tmp/sf/**/stockfish*", recursive=True) if os.path.isfile(f) and not f.endswith((".tar", ".txt", ".md")) and os.access(f, os.X_OK))
sh("sh", "-c", f"echo quit | {ENGINE} | head -1 | tee {OUT}/stockfish_version.txt")
sh("sha256sum", "/tmp/sf/sf.tar")
if BETAS.strip():
    sh("git", "clone", "--depth", "1", "https://github.com/CSSLab/maia3.git", "/tmp/maia3")
    if MAIA3_COMMIT:
        sh("git", "-C", "/tmp/maia3", "fetch", "--depth", "1", "origin", MAIA3_COMMIT)
        sh("git", "-C", "/tmp/maia3", "checkout", MAIA3_COMMIT)
    sh("git", "-C", "/tmp/maia3", "log", "-1", "--format=maia3 commit %h %ad")
    os.makedirs("/tmp/weights", exist_ok=True)
    sh("curl", "-fsSL", "-o", f"/tmp/weights/maia3-{MODEL}.pt", f"https://huggingface.co/UofTCSSLab/Maia3-{MODEL.upper()}/resolve/main/maia3-{MODEL}.pt")
    sh("sha256sum", f"/tmp/weights/maia3-{MODEL}.pt")

## Step 2. The matches

In [ ]:
LADDER = [f"sf-{n}=sf:nodes={n}" for n in NODES.split(",")]
BOTS = [f"bot-beta{b}=bot:maia3={MODEL}+history:beta={b}:rating={RATING}:nodes={BOT_NODES}" for b in BETAS.split(",") if b.strip()]
sh(*CLI, "humanlike-arena", "--engine", ENGINE, "--workers", str(WORKERS), "--players", *LADDER, *BOTS, "--starts-from", HUMAN, "--starts", str(STARTS),
   "--maia3-repo", "/tmp/maia3", "--weights-dir", "/tmp/weights", "--games-out", f"{OUT}/games_arena.jsonl", "--anchor", f"sf-{NODES.split(',')[0]}", "--out", f"{OUT}/arena_elo.md")